In [2]:
import gradio as gr
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

# 1. Define The GAN Architecture
class Generator(nn.Module):
    def __init__(self, input_dim=28, noise_dim=16):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim + noise_dim + 1, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, z, x, y_fake):
        return self.model(torch.cat([z, x, y_fake], dim=1))

# 2. Load The Model (Optimized for CPU)
device = torch.device("cpu")
netG = Generator().to(device)
try:
    netG.load_state_dict(torch.load("netG.pth", map_location=device))
    netG.eval()
except:
    print("Warning: netG.pth not found. Please upload your model file.")

# 3. The Predication Function
def predict_ffr(angle, stenosis, biased_ffr):
    # Create the 28-feature vector (Fill others with defaults)
    # We use the 2 key features you identified: Angle and Stenosis
    features = np.zeros((1, 28))
    features[0, 0] = angle
    features[0, 1] = stenosis

    X_tensor = torch.FloatTensor(features).to(device)
    y_fake_tensor = torch.FloatTensor([[biased_ffr]]).to(device)

    # Fixed seed for reproducibility on Hugging Face
    torch.manual_seed(42)
    z = torch.randn(1, 16).to(device)

    with torch.no_grad():
        y_raw = netG(z, X_tensor, y_fake_tensor).cpu().numpy()[0][0]

    # Physiological Rescaling (0.4 - 1.0)
    y_adj = 0.4 + (0.6 * y_raw)
    y_adj = np.clip(y_adj, 0.4, 1.0)

    delta = y_adj - biased_ffr
    status = "🔴 ISCHEMIC (Significant)" if y_adj <= 0.80 else "🟢 NON-ISCHEMIC (Stable)"

    return f"{y_adj:.3f}", status, f"{delta:+.3f} (GAN Correction Applied)"

# 4. The GRADIO Interface (The "Clinical" Look)
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏥 FFR-GAN: Physiological Bias Correction Prototype")
    gr.Markdown("### Translational Feasibility Demo for Coronary Artery Disease Assessment")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 1. Clinical Inputs")
            angle = gr.Slider(30, 90, value=60, label="Vessel Angle (Degrees)")
            stenosis = gr.Slider(20, 90, value=65, label="Stenosis Severity (%)")
            biased_ffr = gr.Number(value=0.88, label="Biased FFR (Area-based)")
            btn = gr.Button("Run Physiological Adjustment", variant="primary")

        with gr.Column():
            gr.Markdown("### 2. GAN Adjusted Output")
            out_ffr = gr.Textbox(label="Corrected FFR (Diameter-equivalent)")
            out_status = gr.Textbox(label="Clinical Status")
            out_delta = gr.Label(label="Adjustment Delta")

    btn.click(predict_ffr, inputs=[angle, stenosis, biased_ffr], outputs=[out_ffr, out_status, out_delta])

    gr.Markdown("---")
    gr.Markdown("*Note: This is a research prototype for demonstrating translational feasibility. Not for clinical diagnostic use.*")

demo.launch()

/tmp/ipython-input-479/4101356160.py:59: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a79650bd82d2debf41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
